In [24]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
QUADRANT_URL = os.getenv("QUADRANT_URL")
QUADRANT_API_KEY= os.getenv("QUADRANT_API_KEY")

dict = {
    groq_api_key : "GROQ API KEY",
    QUADRANT_URL : "QUADRANT_URL",
    QUADRANT_API_KEY : "QUADRANT_API_KEY"
}

for i in [groq_api_key , QUADRANT_URL , QUADRANT_API_KEY]:
    if groq_api_key:
        print(f"{dict[i]} loaded successfully.")
    else:
        print(f"Could not find {dict[i]}  in the .env file.")

GROQ API KEY loaded successfully.
QUADRANT_URL loaded successfully.
QUADRANT_API_KEY loaded successfully.


In [25]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage


llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


In [26]:
question = HumanMessage('tell me about the Health in 3 points')
system = SystemMessage('You are Teacher. You answer in short sentences.')


messages = [system, question]
response = llm.invoke(messages)


print(response.content)

<think>
Okay, the user asked for "Health in 3 points." Let me break this down. First, I need to define health in a concise way. The World Health Organization's definition is a good start—physical, mental, and social well-being. That's a solid first point.

Next, the user might be looking for key factors that contribute to health. Balanced nutrition is essential because what we eat affects our bodies. Regular exercise is another pillar, as it helps maintain physical health and mental well-being. These two points cover the basics of a healthy lifestyle.

For the third point, mental health is crucial. Stress management and emotional balance are important for overall health. Including mental health ensures the answer is comprehensive. I should also mention prevention and regular check-ups to emphasize proactive health care. That covers the three main areas: definition, key factors, and mental health aspects. Let me make sure each point is clear and concise without being too technical.
</th

In [27]:
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        PromptTemplate,
                                        ChatPromptTemplate
                                        )


In [28]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url=QUADRANT_URL,
    api_key=QUADRANT_API_KEY,
)

print(qdrant_client.get_collections())


collections=[]


In [29]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4769.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
from qdrant_client.models import VectorParams, Distance

qdrant_client.recreate_collection(
    collection_name="nutrition_rag",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print(qdrant_client.get_collections())

/tmp/ipykernel_380/2562890727.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


collections=[CollectionDescription(name='nutrition_rag')]


In [31]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
loader = WebBaseLoader(
    web_paths=("https://www.healthline.com/nutrition/staple-foods-to-make-healthy-eating-easy-all-week-long#chicken",),
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")
print(docs)

Total characters: 20468
[Document(metadata={'source': 'https://www.healthline.com/nutrition/staple-foods-to-make-healthy-eating-easy-all-week-long#chicken', 'title': 'Meal Prep Like a Pro With These 15 Staple Foods', 'description': "Here's how to always have something to make for dinner.", 'language': 'en'}, page_content="\n\nMeal Prep Like a Pro With These 15 Staple Foods\n\nHealthlineHealth ConditionsHealth ConditionsAllBreast CancerCancer CareCaregiving for Alzheimer's DiseaseChronic Kidney DiseaseChronic Obstructive Pulmonary Disease (COPD)Digestive HealthEye HealthHeart HealthMenopauseMental HealthMigraineMultiple Sclerosis (MS)Parkinson’s DiseasePsoriasisRheumatoid Arthritis (RA)Sleep HealthType 2 DiabetesWeight ManagementCondition SpotlightAllControlling Ulcerative ColitisNavigating Life with Bipolar DisorderMastering Geographic AtrophyManaging Type 2 DiabetesWellnessWellness TopicsAllCBDFitnessHealthy AgingHearingMental Well-BeingNutritionParenthoodRecipesSexual HealthSkin Care

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(docs)
texts = [doc.page_content for doc in chunks]

vectors = model.encode(texts)

In [33]:
vectors

array([[-0.01272958, -0.02704687, -0.04238744, ..., -0.01183318,
         0.01501446, -0.0092982 ],
       [-0.01769031, -0.01053934,  0.02613864, ..., -0.02363382,
         0.02421782,  0.08858079],
       [ 0.06632195, -0.07982408, -0.03251132, ..., -0.03757659,
        -0.0441448 ,  0.01739546],
       ...,
       [-0.03831528, -0.0069708 , -0.04734706, ..., -0.04978926,
         0.04907534,  0.02226244],
       [ 0.02501525,  0.01018794, -0.03520567, ..., -0.06983206,
         0.05866154,  0.0416552 ],
       [ 0.00414329, -0.01558778, -0.02669036, ..., -0.07956035,
         0.0625848 ,  0.02577205]], shape=(52, 384), dtype=float32)

In [34]:
len(vectors[0]) , vectors[0]

(384,
 array([-1.27295814e-02, -2.70468667e-02, -4.23874408e-02, -7.23893801e-03,
        -1.98559985e-02, -1.57065708e-02, -1.31032448e-02,  9.84228682e-03,
        -1.13313468e-02,  5.56187006e-03,  1.02526009e-01, -2.63293758e-02,
        -3.72314081e-02, -9.23226327e-02,  2.25567650e-02, -3.78714092e-02,
         1.70342088e-01, -4.81681041e-02, -1.21480385e-02, -5.90427332e-02,
        -6.98740035e-02, -4.34248634e-02,  1.71579160e-02,  3.60988826e-02,
         3.09888297e-03, -1.66930575e-02,  2.15900894e-02, -2.67972983e-02,
        -9.08616483e-02,  4.07227874e-03,  2.16349941e-02, -6.14029355e-02,
         4.82118018e-02, -3.28506082e-02,  1.90974642e-02,  1.71613880e-03,
         3.10746133e-02, -5.43360375e-02, -2.01399196e-02,  1.94954164e-02,
         2.84734927e-02, -8.21642354e-02, -3.94949615e-02, -8.72477964e-02,
         1.18383933e-02, -5.25705405e-02, -1.07205547e-01,  6.22860752e-02,
         1.84469856e-02,  1.06050242e-02, -8.50536153e-02, -1.58929061e-02,
      

In [35]:
from qdrant_client.models import PointStruct

points = []

for i, (text, vector) in enumerate(zip(texts, vectors)):
    points.append(
        PointStruct(
            id=i,
            vector=vector.tolist(),  # convert numpy → list
            payload={"text": text}   # store original chunk
        )
    )

In [36]:
points[:2]

[PointStruct(id=0, vector=[-0.012729581445455551, -0.027046866714954376, -0.04238744080066681, -0.007238938007503748, -0.01985599845647812, -0.01570657081902027, -0.013103244826197624, 0.009842286817729473, -0.011331346817314625, 0.00556187005713582, 0.10252600908279419, -0.02632937580347061, -0.037231408059597015, -0.09232263267040253, 0.022556765004992485, -0.037871409207582474, 0.1703420877456665, -0.04816810414195061, -0.012148038484156132, -0.059042733162641525, -0.06987400352954865, -0.04342486336827278, 0.017157915979623795, 0.03609888255596161, 0.003098882967606187, -0.016693057492375374, 0.021590089425444603, -0.026797298341989517, -0.09086164832115173, 0.004072278738021851, 0.021634994074702263, -0.06140293553471565, 0.048211801797151566, -0.032850608229637146, 0.01909746415913105, 0.001716138795018196, 0.031074613332748413, -0.05433603748679161, -0.020139919593930244, 0.019495416432619095, 0.028473492711782455, -0.0821642354130745, -0.03949496150016785, -0.08724779635667801,

In [37]:
qdrant_client.upsert(
    collection_name="nutrition_rag",
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [38]:
qdrant_client.count(collection_name="nutrition_rag")

CountResult(count=52)

In [50]:
query = "foods for Reduce Fat ?"

query_vector = model.encode(query).tolist()

# print(query_vector)

results = qdrant_client.query_points(
    collection_name="nutrition_rag",
    query=query_vector,
    limit=5
)

results

QueryResponse(points=[ScoredPoint(id=28, version=1, score=0.44043466, payload={'text': 'the next day — but even a small amount of extra dry quick-cooking oats can get put to good use. Sprinkle some into muffins or add it to meatloaf for sturdiness.Health benefitsOats take their place in the pantheon of so-called “superfoods” for good reason. Their soluble fiber has been linked to reduced cholesterol, while their beta glucan can help stabilize blood sugar. Meanwhile, diets rich in whole grains (like oats) may lower the risk of colorectal cancer.Bone brothPrep suggestionsVeggie,'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=46, version=1, score=0.4294124, payload={'text': 'fatty acids, olive oil and health status: a systematic review and meta-analysis of cohort studies.https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4198773/Tosh S. (2013). Review of human studies investigating the post-prandial blood-glucose lowering ability of oat and barley food products.https://pubmed

In [52]:
# For Debugging
# print(results.points)
# get the whole context
context = ""
for i in results.points:
    context += i.payload["text"]

In [65]:
context

"the next day — but even a small amount of extra dry quick-cooking oats can get put to good use. Sprinkle some into muffins or add it to meatloaf for sturdiness.Health benefitsOats take their place in the pantheon of so-called “superfoods” for good reason. Their soluble fiber has been linked to reduced cholesterol, while their beta glucan can help stabilize blood sugar. Meanwhile, diets rich in whole grains (like oats) may lower the risk of colorectal cancer.Bone brothPrep suggestionsVeggie,fatty acids, olive oil and health status: a systematic review and meta-analysis of cohort studies.https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4198773/Tosh S. (2013). Review of human studies investigating the post-prandial blood-glucose lowering ability of oat and barley food products.https://pubmed.ncbi.nlm.nih.gov/23422921/U.S. Department of Agriculture, Agricultural Research Service. (2019). FoodData Central.http://fdc.nal.usda.gov/Wang Q, et al. (2020). Tofu intake is inversely associatedBased15

In [53]:
system = SystemMessagePromptTemplate.from_template("""
You are a certified nutrition assistant.

Use ONLY the provided context to answer the question.
If the answer is not in the context, say:
"I don’t have enough information to answer that."

Guidelines:
- Be accurate and evidence-based
- Do not hallucinate
- Keep answers clear and structured
- If relevant, suggest foods, nutrients, and simple tips
- Consider dietary restrictions if mentioned (vegetarian, vegan, etc.)
- For medical conditions, give general guidance only (not diagnosis)

Context:
{context}
""")

question_prompt = HumanMessagePromptTemplate.from_template("""
Question:
{question}
""")


In [ ]:
messages = [system, question_prompt]

template = ChatPromptTemplate.from_messages(messages)


In [66]:
messages = [system, question_prompt]

template = ChatPromptTemplate.from_messages(messages)


from langchain_core.output_parsers import StrOutputParser

chain = template | llm | StrOutputParser()
result = chain.invoke({
    "context": context,   # RAG output (joined text)
    "question": query
})

print(result)

<think>
Okay, the user is asking about foods that help reduce fat. Let me look through the provided context to find relevant information.

First, the context mentions oats and their health benefits, like soluble fiber reducing cholesterol and beta-glucan stabilizing blood sugar. That's good for reducing fat, especially LDL cholesterol. So oats should be included.

Then there's bone broth, but the context doesn't specify how it helps with fat reduction. Maybe it's more about overall health. Not sure, so maybe skip unless there's more info. Wait, the context also lists other foods like chickpeas, lentils, quinoa, spinach, tomatoes, olive oil, onions, apples. Let me check each.

Chickpeas are high in fiber and protein, which can aid in weight management by keeping you full. The context suggests using them in various dishes. That's a good point.

Salmon is mentioned. It's a fatty fish rich in omega-3s, which can help reduce triglycerides. So that's a positive for fat reduction.

Tofu is in